[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/user/repo/blob/main/notebook.ipynb)

### Lecture 04 Hypothesis test
## - Non-Parametric tests:
- Wilcoxon rank-sum
- Kruskal–Wallis
- Kolmogorov–Smirnov

In [1]:
import numpy as np
import pandas as pd
from scipy import stats

import matplotlib.pyplot as plt

- again, starting by organizing data

In [3]:
### start by loading data and assign month and year. Here we are using Winter Niño3.4
df = pd.read_csv("/content/n34_all.csv", parse_dates=["time"])
df['year'] = df['time'].dt.year
df['month'] = df['time'].dt.month
df

,time,nino34,year,month
0,1854-01-01,25.897331,1854,1
1,1854-02-01,26.055464,1854,2
2,1854-03-01,26.908459,1854,3
3,1854-04-01,27.631338,1854,4
4,1854-05-01,27.902256,1854,5
...,...,...,...,...
2067,2026-04-01,28.126972,2026,4
2068,2026-05-01,28.838810,2026,5
2069,2026-06-01,29.267714,2026,6
2070,2026-07-01,29.277988,2026,7


- get djf

In [4]:
### use data from December, January, and February ----
djf = df[df['month'].isin([12, 1, 2])].copy()
djf['djf_year'] = djf['year']
djf.loc[djf['month'] == 12, 'djf_year'] = djf['year'] + 1  # Dec belongs to following winter
djf

,time,nino34,year,month,djf_year
0,1854-01-01,25.897331,1854,1,1854
1,1854-02-01,26.055464,1854,2,1854
11,1854-12-01,25.118660,1854,12,1855
12,1855-01-01,25.940777,1855,1,1855
13,1855-02-01,26.953278,1855,2,1855
...,...,...,...,...,...
2052,2025-01-01,25.815334,2025,1,2025
2053,2025-02-01,26.332956,2025,2,2025
2063,2025-12-01,25.928493,2025,12,2026
2064,2026-01-01,25.970411,2026,1,2026


- same 30 years period

In [5]:
### make sure we use only years with complete DFJ
counts = djf.groupby('djf_year')['nino34'].count()
complete_years = counts[counts == 3].index  # keep only complete D-J-F data
djf_seasonal = djf[djf['djf_year'].isin(complete_years)].groupby('djf_year')['nino34'].mean().reset_index()

### Define the two periods
early = djf_seasonal.loc[(djf_seasonal["djf_year"] >= 1951) & (djf_seasonal["djf_year"] <= 1980), "nino34"].values
recent = djf_seasonal.loc[(djf_seasonal["djf_year"] >= 1996) & (djf_seasonal["djf_year"] <= 2025), "nino34"].values


###  Wilcoxon rank-sum
stats.ranksums(x,y)

In [6]:
z_stat, p_val = stats.ranksums(early, recent)
print (z_stat, p_val)

-0.07392212709545729 0.9410723491398268


The normal approximation for rank-sum is often improved by adding a continuity correction C =0.5

$$
z = \frac{R_x - E[R_x] + C}{\sqrt{\text{Var}[R_x]}} \quad \text{where} \quad C =
\begin{cases}
-0.5 & \text{for } R_x > \hat{\mu}_{[R_x]} \\
0.5 & \text{for } R_x < \hat{\mu}_{[R_x]}
\end{cases}
$$

and in python, this is done using stats.mannwhitneyu

In [7]:
u_stat, p_value = stats.mannwhitneyu(early, recent, alternative='two-sided')
print('mannwhitneyu U_stat',u_stat, 'p-value',p_value)

mannwhitneyu U_stat 445.0 p-value 0.9469559422255942


### Kolmogorov-Smirnov (KS) test
 stats.ks_2samp(x,y)

In [8]:
# ---- KS two-sample test (built-in scipy function) ----
D_stat, p_val = stats.ks_2samp(early, recent)
print (D_stat, p_val)

0.13333333333333333 0.9578462903438838


-- back to the lecture --

### - kruskal Wallis
H_stat, p_value = stats.kruskal(group1, group2, group3, ...)


In [9]:
seasonal = pd.read_csv("/content/seasonal_ppt_tmean_JFM_AMJ_JAS_OND.csv")
seasonal

,Unnamed: 0,year,season,ppt (inches),tmean (degrees F),n_months
0,0,1895,AMJ,7.99,56.133333,3
1,1,1895,JAS,2.87,62.466667,3
2,2,1895,JFM,17.61,41.800000,3
3,3,1895,OND,14.60,46.200000,3
4,4,1896,AMJ,12.18,53.400000,3
...,...,...,...,...,...,...
521,521,2025,JAS,1.37,69.466667,3
522,522,2025,JFM,15.28,43.200000,3
523,523,2025,OND,16.40,49.566667,3
524,524,2026,AMJ,4.68,58.900000,3


In [ ]:
# Explicit series per season
jfm_series = seasonal.loc[seasonal["season"] == "JFM", "ppt (inches)"]
amj_series = seasonal.loc[seasonal["season"] == "AMJ", "ppt (inches)"]
jas_series = seasonal.loc[seasonal["season"] == "JAS", "ppt (inches)"]
ond_series = seasonal.loc[seasonal["season"] == "OND", "ppt (inches)"]

In [ ]:
# Kruskal-Wallis test
h_stat, p_value = stats.kruskal(jfm_series, amj_series, jas_series, ond_series)

print(f"H-statistic: {h_stat:.4f}")
print(f"p-value: {p_value:.4e}")



H-statistic: 411.7659
p-value: 6.2587e-89


-- back to lecture --

### Permutation-test
- back to DJF Nino34

In [ ]:
### Define the two periods
early = djf_seasonal.loc[(djf_seasonal["djf_year"] >= 1951) & (djf_seasonal["djf_year"] <= 1980), "nino34"].values
recent = djf_seasonal.loc[(djf_seasonal["djf_year"] >= 1996) & (djf_seasonal["djf_year"] <= 2025), "nino34"].values


In [ ]:
Nx, Ny = len(early), len(recent)
N = Nx + Ny
combined = np.concatenate([early, recent])
z_obs, p_asymp = stats.ranksums(early, recent)

perm_z = np.zeros(500)

for i in range(500):
    shuffled = np.random.permutation(combined)
    g1, g2 = shuffled[:Nx], shuffled[Nx:]
    z_i, _ = stats.ranksums(g1, g2)
    perm_z[i] = z_i

p_perm = np.mean(np.abs(perm_z) >= np.abs(z_obs))
p_perm

np.float64(0.938)

**Does mean temperature in oregan chnage woth season**
- use kolo to test JMA and OND

- use Kruskal- Wallis test all seasons

In [13]:
df = pd.read_csv("/content/PRISM_ppt_tmean_45.0000_-123.0000_189501_202607.csv", parse_dates=["Date"])
df['year'] = df['Date'].dt.year
df['month'] = df['Date'].dt.month
df

,Date,ppt (inches),tmean (degrees F),year,month
0,1895-01-01,11.60,38.3,1895,1
1,1895-02-01,1.77,42.5,1895,2
2,1895-03-01,4.24,44.6,1895,3
3,1895-04-01,2.71,51.5,1895,4
4,1895-05-01,4.63,55.3,1895,5
...,...,...,...,...,...
1574,2026-03-01,3.09,49.1,2026,3
1575,2026-04-01,3.11,52.7,2026,4
1576,2026-05-01,0.60,59.6,2026,5
1577,2026-06-01,0.97,64.4,2026,6


In [14]:
df_JMA = df[df['month'].isin([4, 5, 6])].copy()
df_JMA['df_year'] = df_JMA['year']
df_JMA

,Date,ppt (inches),tmean (degrees F),year,month,df_year
3,1895-04-01,2.71,51.5,1895,4,1895
4,1895-05-01,4.63,55.3,1895,5,1895
5,1895-06-01,0.65,61.6,1895,6,1895
15,1896-04-01,5.63,47.1,1896,4,1896
16,1896-05-01,5.34,52.4,1896,5,1896
...,...,...,...,...,...,...
1564,2025-05-01,0.88,57.2,2025,5,2025
1565,2025-06-01,0.83,63.8,2025,6,2025
1575,2026-04-01,3.11,52.7,2026,4,2026
1576,2026-05-01,0.60,59.6,2026,5,2026


In [15]:
df_OND = df[df['month'].isin([10, 11, 12])].copy()
df_OND['df_year'] = df_OND['year']
df_OND

,Date,ppt (inches),tmean (degrees F),year,month,df_year
9,1895-10-01,0.01,53.7,1895,10,1895
10,1895-11-01,3.70,43.4,1895,11,1895
11,1895-12-01,10.89,41.5,1895,12,1895
21,1896-10-01,2.53,53.9,1896,10,1896
22,1896-11-01,16.20,39.0,1896,11,1896
...,...,...,...,...,...,...
1558,2024-11-01,7.70,46.0,2024,11,2024
1559,2024-12-01,7.76,42.7,2024,12,2024
1569,2025-10-01,3.20,53.1,2025,10,2025
1570,2025-11-01,5.26,49.0,2025,11,2025


In [27]:
JMA_series = df_JMA['tmean (degrees F)'].values
OND_series = df_OND['tmean (degrees F)'].values


array([51.5, 55.3, 61.6, 47.1, 52.4, 60.7, 53.5, 60.1, 61.1, 51.5, 55.8,
       62.4, 48.1, 52.1, 58.9, 51. , 57.3, 62.3, 48.2, 55.5, 58.3, 48.9,
       55.7, 60.5, 48.3, 55.5, 62.5, 53.5, 57.1, 62.2, 54. , 55.3, 60.5,
       54.8, 56.1, 59.2, 51.9, 58. , 61.6, 50.6, 52.9, 60. , 49.6, 53.6,
       61.6, 52.2, 58.4, 58.9, 47.8, 52.9, 58.5, 48.4, 57.3, 61.4, 49.8,
       55.6, 60.9, 51.7, 58.3, 59.6, 54. , 55.1, 60.5, 51.2, 52.6, 60.1,
       47.4, 53.3, 60. , 51.5, 53.6, 64.7, 50.8, 55. , 58.3, 47. , 52.5,
       60.2, 49.2, 54.9, 62.8, 47.2, 56.6, 65.6, 51.4, 56. , 61.3, 51.3,
       59.4, 63.5, 52.9, 59.3, 62.5, 58. , 57.2, 65.1, 50. , 53.3, 62.9,
       49.6, 59.4, 60.8, 47.8, 56.3, 60.8, 53.3, 53.1, 59. , 53.1, 60. ,
       60.9, 50.7, 54.9, 63.7, 49.9, 52.1, 60.2, 57.2, 59.2, 62.7, 50.2,
       55.1, 62.7, 53.8, 58.8, 62.8, 49. , 57. , 63. , 51.7, 57.3, 63.5,
       53.3, 57.7, 59.8, 52.7, 59.6, 64.4, 53. , 56.8, 61.7, 51.5, 55.5,
       59.9, 53.1, 54.5, 60. , 50.2, 56.2, 60.8, 49

In [25]:
D_stat, p_val = stats.ks_2samp(JMA_series, OND_series)
print (D_stat, p_val)

0.5781478911249903 1.4765678484349703e-61


In [26]:
h_stat, p_value = stats.kruskal(JMA_series, OND_series)

print(f"H-statistic: {h_stat:.4f}")
print(f"p-value: {p_value:.4e}")



H-statistic: 327.0717
p-value: 4.1742e-73
